In [1]:
import shutil, os
if os.path.exists('/content/drive'):
    shutil.rmtree('/content/drive')
    print("cleared")
os.makedirs('/content/drive', exist_ok=True)
print("fresh mountpoint ready")

fresh mountpoint ready


In [2]:
from google.colab import drive
import time
drive.mount('/content/drive')
time.sleep(20)

import os
print(os.listdir('/content/drive/MyDrive/AARN_project'))

Mounted at /content/drive
['dataset', 'notebooks', 'models', 'results', 'models_new', 'results_new']


In [3]:
import os

mount_point = '/content/drive'

if os.path.exists(mount_point) and os.path.isdir(mount_point):
    if os.listdir(mount_point):
        print(f"Google Drive is likely mounted at '{mount_point}' and contains files.")
    else:
        print(f"Google Drive mount point '{mount_point}' exists but is empty.")
else:
    print(f"Google Drive is not mounted at '{mount_point}'.")

Google Drive is likely mounted at '/content/drive' and contains files.


In [3]:
# ── CELL 1 — Imports + Paths ─────────────────────────────────────────────────
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import cohen_kappa_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt
import rasterio
from rasterio.transform import xy as rio_xy, rowcol as rio_rowcol
from pyproj import Transformer
import glob, gc, os, csv

BASE      = '/content/drive/MyDrive/AARN_project'
IN_DIR    = f'{BASE}/dataset/processed_new'
DATA_DIR  = f'{BASE}/dataset/processed/data'
MODEL_DIR = f'{BASE}/models_new'
RES_DIR   = f'{BASE}/results_new'
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(RES_DIR,   exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

# Paper Table 3 exact hyperparameters
HYP = dict(
    lr=0.001, weight_decay=1e-4, batch_size=32, epochs=200,
    n_heads=5, kernel_size=3, dropout=0.3,
    patience=25, grad_clip=1.0, seed=42,
)

S2_GLOB = {
    'arkansas'  : f'{DATA_DIR}/Sentinel2_2021_36steps_Arkansas*.tif',
    'california': f'{DATA_DIR}/Sentinel2_2021_36steps_California_Sacramento*.tif',
}
CDL_PATH = {
    'arkansas'  : f'{DATA_DIR}/CDL_2021_Labels_Arkansas.tif',
    'california': f'{DATA_DIR}/CDL_2021_Labels_California_Sacramento.tif',
}

torch.manual_seed(HYP['seed'])
np.random.seed(HYP['seed'])
print("✅ Cell 1 done")


Device: cpu
✅ Cell 1 done


In [4]:
# ── CELL 2 — MCTNet Architecture (Wang et al. 2024) ──────────────────────────
class ECA(nn.Module):
    def __init__(self, channels, kernel=3):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        self.conv     = nn.Conv1d(1, 1, kernel_size=kernel,
                                  padding=kernel//2, bias=False)
        self.sigmoid  = nn.Sigmoid()

    def forward(self, x):
        y = self.avg_pool(x.transpose(1, 2))
        y = self.conv(y.transpose(1, 2))
        y = self.sigmoid(y)
        return x * y


class ALPE(nn.Module):
    """Eq. 3: ALPE(t) = ECA(Conv1D(PE(t) x mask)) — Stage 1 only."""
    def __init__(self, seq_len=36, d_model=10, kernel=3):
        super().__init__()
        self.conv = nn.Conv1d(d_model, d_model, kernel_size=kernel,
                               padding=kernel//2, bias=False)
        self.eca  = ECA(d_model, kernel=kernel)
        pe  = torch.zeros(seq_len, d_model)
        pos = torch.arange(seq_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() *
                        (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div[:d_model // 2])
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x, mask):
        pe        = self.pe.expand(x.size(0), -1, -1)
        masked_pe = pe * mask.unsqueeze(-1)
        masked_pe = self.conv(masked_pe.transpose(1,2)).transpose(1,2)
        masked_pe = self.eca(masked_pe)
        return x + masked_pe


class CNNSubModule(nn.Module):
    def __init__(self, in_ch, out_ch, kernel=3):
        super().__init__()
        self.conv1 = nn.Conv1d(in_ch,  out_ch, kernel, padding=kernel//2, bias=False)
        self.bn1   = nn.BatchNorm1d(out_ch)
        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel, padding=kernel//2, bias=False)
        self.bn2   = nn.BatchNorm1d(out_ch)
        self.skip  = nn.Conv1d(in_ch, out_ch, 1, bias=False) \
                     if in_ch != out_ch else nn.Identity()

    def forward(self, x):
        res = self.skip(x)
        x   = F.relu(self.bn1(self.conv1(x)))
        x   = self.bn2(self.conv2(x))
        return F.relu(x + res)


class TransformerSubModule(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.3):
        super().__init__()
        self.attn    = nn.MultiheadAttention(d_model, n_heads,
                                              dropout=dropout, batch_first=True)
        self.ffn     = nn.Sequential(
            nn.Linear(d_model, d_model * 2), nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 2, d_model),
        )
        self.norm1   = nn.LayerNorm(d_model)
        self.norm2   = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        a, _ = self.attn(x, x, x)
        x = self.norm1(x + self.dropout(a))
        x = self.norm2(x + self.dropout(self.ffn(x)))
        return x


class CTFusionStage(nn.Module):
    def __init__(self, in_ch, out_ch, n_heads, kernel=3, dropout=0.3,
                 use_alpe=False, seq_len=36):
        super().__init__()
        self.use_alpe = use_alpe
        if use_alpe:
            self.alpe = ALPE(seq_len=seq_len, d_model=in_ch, kernel=kernel)
        self.cnn           = CNNSubModule(in_ch, out_ch, kernel)
        self.trans         = TransformerSubModule(in_ch, n_heads, dropout)
        self.fusion_linear = nn.Linear(out_ch + in_ch, out_ch)
        self.pool          = nn.MaxPool1d(2)

    def forward(self, x, mask=None):
        x_t   = self.alpe(x, mask) if (self.use_alpe and mask is not None) else x
        x_cnn = self.cnn(x.transpose(1,2)).transpose(1,2)
        x_t   = self.trans(x_t)
        fused = self.fusion_linear(torch.cat([x_cnn, x_t], dim=-1))
        return self.pool(fused.transpose(1,2)).transpose(1,2)


class MCTNet(nn.Module):
    def __init__(self, n_classes=5, seq_len=36, n_bands=10,
                 n_heads=5, kernel=3, dropout=0.3):
        super().__init__()
        self.stage1     = CTFusionStage(n_bands, 20, n_heads, kernel, dropout,
                                         use_alpe=True, seq_len=seq_len)
        self.stage2     = CTFusionStage(20, 40, n_heads, kernel, dropout)
        self.stage3     = CTFusionStage(40, 80, n_heads, kernel, dropout)
        self.classifier = nn.Linear(80, n_classes)
        self.dropout    = nn.Dropout(dropout)

    def forward(self, x, mask):
        x = self.stage1(x, mask)
        x = self.stage2(x)
        x = self.stage3(x)
        x = x.max(dim=1).values
        x = self.dropout(x)
        return self.classifier(x)


_tmp = MCTNet(n_classes=5)
print(f"MCTNet parameters: {sum(p.numel() for p in _tmp.parameters() if p.requires_grad):,}  (paper: 54,428)")
del _tmp
print("✅ Cell 2 done — MCTNet defined")


MCTNet parameters: 73,578  (paper: 54,428)
✅ Cell 2 done — MCTNet defined


In [6]:
# ── CELL 3 — Dataset, training loop, RAM-safe test evaluation ────────────────
class CropDataset(Dataset):
    def __init__(self, X, mask, y, indices):
        self.X    = torch.tensor(X[indices],    dtype=torch.float32)
        self.mask = torch.tensor(mask[indices], dtype=torch.float32)
        self.y    = torch.tensor(y[indices],    dtype=torch.long)
    def __len__(self):       return len(self.y)
    def __getitem__(self, i): return self.X[i], self.mask[i], self.y[i]


def compute_class_weights(y_train, n_classes):
    counts  = np.bincount(y_train, minlength=n_classes).astype(np.float32)
    weights = 1.0 / (counts + 1e-6)
    return torch.tensor(weights / weights.sum() * n_classes,
                        dtype=torch.float32).to(DEVICE)


def train_model(model, train_loader, val_loader, region, class_weights):
    optimizer = torch.optim.Adam(model.parameters(),
                                  lr=HYP['lr'], weight_decay=HYP['weight_decay'])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                    optimizer, T_max=HYP['epochs'])
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    best_val, patience_c = float('inf'), 0
    t_losses, v_losses   = [], []

    for epoch in range(HYP['epochs']):
        model.train()
        t_loss = 0
        for X, mask, y in train_loader:
            X, mask, y = X.to(DEVICE), mask.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(X, mask), y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), HYP['grad_clip'])
            optimizer.step()
            t_loss += loss.item()
        t_loss /= len(train_loader)

        model.eval()
        v_loss = 0
        with torch.no_grad():
            for X, mask, y in val_loader:
                X, mask, y = X.to(DEVICE), mask.to(DEVICE), y.to(DEVICE)
                v_loss += criterion(model(X, mask), y).item()
        v_loss /= len(val_loader)

        scheduler.step()
        t_losses.append(t_loss)
        v_losses.append(v_loss)

        if (epoch + 1) % 20 == 0:
            print(f"  Epoch {epoch+1:3d} | train={t_loss:.4f} | val={v_loss:.4f}")

        if v_loss < best_val:
            best_val, patience_c = v_loss, 0
            torch.save(model.state_dict(),
                       f'{MODEL_DIR}/new_mctnet_{region}_best.pth')
        else:
            patience_c += 1
            if patience_c >= HYP['patience']:
                print(f"  Early stopping at epoch {epoch+1}")
                break

    plt.figure(figsize=(8, 4))
    plt.plot(t_losses, label='Train'); plt.plot(v_losses, label='Val')
    plt.xlabel('Epoch'); plt.ylabel('Loss')
    plt.title(f'MCTNet — {region}'); plt.legend(); plt.tight_layout()
    plt.savefig(f'{RES_DIR}/new_training_curve_{region}.png', dpi=150)
    plt.close()
    print("  Training curve saved.")


def extract_test_batch(locs_batch, s2_tiles, cdl_path):
    """Extract spectral features for one batch of test locations. RAM-safe."""
    N = len(locs_batch)
    X_b    = np.zeros((N, 360), dtype=np.float32)
    filled = np.zeros(N, dtype=bool)

    with rasterio.open(cdl_path) as cdl_ref:
        cdl_tr  = cdl_ref.transform
        cdl_crs = cdl_ref.crs

    xs, ys = rio_xy(cdl_tr, locs_batch[:, 0].tolist(), locs_batch[:, 1].tolist())
    xs, ys = np.array(xs, dtype=np.float64), np.array(ys, dtype=np.float64)

    for tile in s2_tiles:
        if filled.all(): break
        with rasterio.open(tile) as src:
            if cdl_crs.to_epsg() != src.crs.to_epsg():
                t = Transformer.from_crs(cdl_crs.to_epsg(),
                                          src.crs.to_epsg(), always_xy=True)
                xs_t, ys_t = t.transform(xs, ys)
            else:
                xs_t, ys_t = xs, ys
            rows, cols = rio_rowcol(src.transform, xs_t, ys_t)
            rows, cols = np.array(rows), np.array(cols)
            ok = (rows >= 0) & (rows < src.height) & \
                 (cols >= 0) & (cols < src.width) & (~filled)
            for j in np.where(ok)[0]:
                win = rasterio.windows.Window(int(cols[j]), int(rows[j]), 1, 1)
                X_b[j]    = src.read(window=win)[:, 0, 0]
                filled[j] = True

    if X_b.max() > 10: X_b /= 10000.0
    return np.clip(X_b.reshape(N, 36, 10), 0.0, 1.0).astype(np.float32)


def evaluate_on_test(model, region, n_classes, class_names, batch_size=256):
    """Full test set evaluation — extracts spectral in batches, never OOM."""
    test_locs = np.load(f'{IN_DIR}/new_test_locs_{region}.npy')
    test_y    = np.load(f'{IN_DIR}/new_test_y_{region}.npy')
    s2_tiles  = sorted(glob.glob(S2_GLOB[region]))
    N         = len(test_locs)
    print(f"\n  Evaluating {N:,} test pixels (batches of {batch_size})...")

    model.eval()
    all_preds = []

    for start in range(0, N, batch_size):
        end   = min(start + batch_size, N)
        X_b   = extract_test_batch(test_locs[start:end], s2_tiles, CDL_PATH[region])
        mask_b = np.zeros((end-start, 36), dtype=np.float32)
        for t in range(36):
            mask_b[:, t] = (X_b[:, t, :].max(axis=1) > 0).astype(np.float32)
        with torch.no_grad():
            preds = model(
                torch.tensor(X_b,    dtype=torch.float32).to(DEVICE),
                torch.tensor(mask_b, dtype=torch.float32).to(DEVICE)
            ).argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        if (start // batch_size) % 100 == 0:
            print(f"    {end:,}/{N:,}")

    all_preds = np.array(all_preds)
    OA    = (all_preds == test_y).mean()
    Kappa = cohen_kappa_score(test_y, all_preds)
    F1    = f1_score(test_y, all_preds, average='macro')
    CM    = confusion_matrix(test_y, all_preds)

    print(f"\n  ── {region.upper()} Test Results ──")
    print(f"    OA={OA:.4f}  Kappa={Kappa:.4f}  F1={F1:.4f}")
    for i, (name, score) in enumerate(zip(class_names,
                                           f1_score(test_y, all_preds, average=None))):
        print(f"    F1 {name:<14}: {score:.4f}")

    fig, ax = plt.subplots(figsize=(n_classes+1, n_classes))
    ax.imshow(CM, cmap='Blues')
    ax.set_xticks(range(n_classes)); ax.set_yticks(range(n_classes))
    ax.set_xticklabels(class_names, rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(class_names, fontsize=8)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title(f'Confusion Matrix — {region}')
    for i in range(n_classes):
        for j in range(n_classes):
            ax.text(j, i, f'{CM[i,j]:,}', ha='center', va='center', fontsize=6)
    plt.tight_layout()
    plt.savefig(f'{RES_DIR}/new_confusion_matrix_{region}.png', dpi=150)
    plt.close()
    return OA, Kappa, F1


print("✅ Cell 3 done — all utilities defined")


✅ Cell 3 done — all utilities defined


In [15]:
# ── CELL 4 — Arkansas ────────────────────────────────────────────────────────
region      = 'arkansas'
n_classes   = 5
class_names = ['Soybeans', 'Rice', 'Corn', 'Cotton', 'Others']

print(f"\n{'='*60}\nPHASE 1 — MCTNet — {region.upper()}\n{'='*60}")

X    = np.load(f'{IN_DIR}/new_X_{region}.npy')
mask = np.load(f'{IN_DIR}/new_mask_{region}.npy')
y    = np.load(f'{IN_DIR}/new_y_{region}.npy')
tr   = np.load(f'{IN_DIR}/new_train_idx_{region}.npy')
vl   = np.load(f'{IN_DIR}/new_val_idx_{region}.npy')

print(f"Train: {len(tr)}  Val: {len(vl)}")
class_weights = compute_class_weights(y[tr], n_classes)
print(f"Class weights: {class_weights.cpu().numpy().round(3)}")

train_loader = DataLoader(CropDataset(X, mask, y, tr),
                           batch_size=HYP['batch_size'], shuffle=True)
val_loader   = DataLoader(CropDataset(X, mask, y, vl),
                           batch_size=HYP['batch_size'])

model = MCTNet(n_classes=n_classes, seq_len=36, n_bands=10,
               n_heads=HYP['n_heads'], kernel=HYP['kernel_size'],
               dropout=HYP['dropout']).to(DEVICE)
print(f"Parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

train_model(model, train_loader, val_loader, region, class_weights)
model.load_state_dict(torch.load(f'{MODEL_DIR}/new_mctnet_{region}_best.pth'))
OA_ark, K_ark, F1_ark = evaluate_on_test(model, region, n_classes, class_names)

del X, mask, y; gc.collect()



PHASE 1 — MCTNet — ARKANSAS
Train: 1200  Val: 300
Class weights: [1. 1. 1. 1. 1.]
Parameters: 73,578
  Epoch  20 | train=0.0196 | val=0.2289
  Early stopping at epoch 35
  Training curve saved.

  Evaluating 8,500 test pixels (batches of 256)...
    256/8,500

  ── ARKANSAS Test Results ──
    OA=0.9515  Kappa=0.9263  F1=0.9206
    F1 Soybeans      : 0.9560
    F1 Rice          : 0.9807
    F1 Corn          : 0.9429
    F1 Cotton        : 0.9072
    F1 Others        : 0.8162


12

In [7]:
# ── CELL 5 — California ───────────────────────────────────────────────────────
region      = 'california'
n_classes   = 6
class_names = ['Grapes', 'Rice', 'Alfalfa', 'Almonds', 'Pistachios', 'Others']

print(f"\n{'='*60}\nPHASE 1 — MCTNet — {region.upper()}\n{'='*60}")

X    = np.load(f'{IN_DIR}/new_X_{region}.npy')
mask = np.load(f'{IN_DIR}/new_mask_{region}.npy')
y    = np.load(f'{IN_DIR}/new_y_{region}.npy')
tr   = np.load(f'{IN_DIR}/new_train_idx_{region}.npy')
vl   = np.load(f'{IN_DIR}/new_val_idx_{region}.npy')

print(f"Train: {len(tr)}  Val: {len(vl)}")
class_weights = compute_class_weights(y[tr], n_classes)

train_loader = DataLoader(CropDataset(X, mask, y, tr),
                           batch_size=HYP['batch_size'], shuffle=True)
val_loader   = DataLoader(CropDataset(X, mask, y, vl),
                           batch_size=HYP['batch_size'])

model = MCTNet(n_classes=n_classes, seq_len=36, n_bands=10,
               n_heads=HYP['n_heads'], kernel=HYP['kernel_size'],
               dropout=HYP['dropout']).to(DEVICE)

train_model(model, train_loader, val_loader, region, class_weights)
model.load_state_dict(torch.load(f'{MODEL_DIR}/new_mctnet_{region}_best.pth'))
OA_cal, K_cal, F1_cal = evaluate_on_test(model, region, n_classes, class_names)

del X, mask, y; gc.collect()




PHASE 1 — MCTNet — CALIFORNIA
Train: 1440  Val: 360
  Epoch  20 | train=0.0327 | val=0.4349
  Early stopping at epoch 38
  Training curve saved.

  Evaluating 8,200 test pixels (batches of 256)...
    256/8,200

  ── CALIFORNIA Test Results ──
    OA=0.9340  Kappa=0.9121  F1=0.9123
    F1 Grapes        : 0.9440
    F1 Rice          : 0.9923
    F1 Alfalfa       : 0.9288
    F1 Almonds       : 0.7923
    F1 Pistachios    : 0.8899
    F1 Others        : 0.9264


27

In [9]:
# Cell 6 — fix: hardcode Arkansas results since they were lost in reconnect
OA_ark, K_ark, F1_ark = 0.9515, 0.9263, 0.9206
OA_cal, K_cal, F1_cal = 0.9340, 0.9121, 0.9123

print("\n" + "="*60)
print("RESULTS vs PAPER — Wang et al. 2024 Table 5")
print("="*60)

paper = {
    'arkansas':   {'OA': 0.968, 'Kappa': 0.951, 'F1': 0.933},
    'california': {'OA': 0.852, 'Kappa': 0.806, 'F1': 0.829},
}
ours = {
    'arkansas':   {'OA': OA_ark, 'Kappa': K_ark, 'F1': F1_ark},
    'california': {'OA': OA_cal, 'Kappa': K_cal, 'F1': F1_cal},
}

print(f"\n{'Region':<12} {'Metric':<8} {'Paper':>8} {'Ours':>8} {'Delta':>8}")
print("-"*44)
for region in ['arkansas', 'california']:
    for metric in ['OA', 'Kappa', 'F1']:
        p = paper[region][metric]; o = ours[region][metric]; d = o - p
        print(f"{region:<12} {metric:<8} {p:>8.3f} {o:>8.3f} {'+' if d>=0 else ''}{d:>7.3f}")
    print()


RESULTS vs PAPER — Wang et al. 2024 Table 5

Region       Metric      Paper     Ours    Delta
--------------------------------------------
arkansas     OA          0.968    0.952  -0.016
arkansas     Kappa       0.951    0.926  -0.025
arkansas     F1          0.933    0.921  -0.012

california   OA          0.852    0.934 +  0.082
california   Kappa       0.806    0.912 +  0.106
california   F1          0.829    0.912 +  0.083



In [6]:
# CELL 6b — Reload all functions after session reset

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import cohen_kappa_score, f1_score
import matplotlib.pyplot as plt
import glob, gc, os, csv

BASE      = '/content/drive/MyDrive/AARN_project'
IN_DIR    = f'{BASE}/dataset/processed_new'
MODEL_DIR = f'{BASE}/models_new'
RES_DIR   = f'{BASE}/results_new'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

# ── MCTNet classes ────────────────────────────────────────────────────────────
class ECA(nn.Module):
    def __init__(self, channels, kernel=3):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        self.conv     = nn.Conv1d(1, 1, kernel_size=kernel,
                                  padding=kernel//2, bias=False)
        self.sigmoid  = nn.Sigmoid()
    def forward(self, x):
        y = self.avg_pool(x.transpose(1,2))
        y = self.conv(y.transpose(1,2))
        return x * self.sigmoid(y)

class ALPE(nn.Module):
    def __init__(self, seq_len=36, d_model=10, kernel=3):
        super().__init__()
        self.conv = nn.Conv1d(d_model, d_model, kernel_size=kernel,
                               padding=kernel//2, bias=False)
        self.eca  = ECA(d_model, kernel=kernel)
        pe  = torch.zeros(seq_len, d_model)
        pos = torch.arange(seq_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() *
                        (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div[:d_model//2])
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x, mask):
        pe        = self.pe.expand(x.size(0), -1, -1)
        masked_pe = pe * mask.unsqueeze(-1)
        masked_pe = self.conv(masked_pe.transpose(1,2)).transpose(1,2)
        return x + self.eca(masked_pe)

class CNNSubModule(nn.Module):
    def __init__(self, in_ch, out_ch, kernel=3):
        super().__init__()
        self.conv1 = nn.Conv1d(in_ch,  out_ch, kernel, padding=kernel//2, bias=False)
        self.bn1   = nn.BatchNorm1d(out_ch)
        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel, padding=kernel//2, bias=False)
        self.bn2   = nn.BatchNorm1d(out_ch)
        self.skip  = nn.Conv1d(in_ch, out_ch, 1, bias=False) \
                     if in_ch != out_ch else nn.Identity()
    def forward(self, x):
        res = self.skip(x)
        x   = F.relu(self.bn1(self.conv1(x)))
        x   = self.bn2(self.conv2(x))
        return F.relu(x + res)

class TransformerSubModule(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.3):
        super().__init__()
        self.attn    = nn.MultiheadAttention(d_model, n_heads,
                                              dropout=dropout, batch_first=True)
        self.ffn     = nn.Sequential(
            nn.Linear(d_model, d_model*2), nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model*2, d_model),
        )
        self.norm1   = nn.LayerNorm(d_model)
        self.norm2   = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        a, _ = self.attn(x, x, x)
        x = self.norm1(x + self.dropout(a))
        x = self.norm2(x + self.dropout(self.ffn(x)))
        return x

class CTFusionStage(nn.Module):
    def __init__(self, in_ch, out_ch, n_heads, kernel=3, dropout=0.3,
                 use_alpe=False, seq_len=36):
        super().__init__()
        self.use_alpe = use_alpe
        if use_alpe:
            self.alpe = ALPE(seq_len=seq_len, d_model=in_ch, kernel=kernel)
        self.cnn           = CNNSubModule(in_ch, out_ch, kernel)
        self.trans         = TransformerSubModule(in_ch, n_heads, dropout)
        self.fusion_linear = nn.Linear(out_ch + in_ch, out_ch)
        self.pool          = nn.MaxPool1d(2)
    def forward(self, x, mask=None):
        x_t   = self.alpe(x, mask) if (self.use_alpe and mask is not None) else x
        x_cnn = self.cnn(x.transpose(1,2)).transpose(1,2)
        x_t   = self.trans(x_t)
        fused = self.fusion_linear(torch.cat([x_cnn, x_t], dim=-1))
        return self.pool(fused.transpose(1,2)).transpose(1,2)

class MCTNet(nn.Module):
    def __init__(self, n_classes=5, seq_len=36, n_bands=10,
                 n_heads=5, kernel=3, dropout=0.3):
        super().__init__()
        self.stage1     = CTFusionStage(n_bands, 20, n_heads, kernel, dropout,
                                         use_alpe=True, seq_len=seq_len)
        self.stage2     = CTFusionStage(20, 40, n_heads, kernel, dropout)
        self.stage3     = CTFusionStage(40, 80, n_heads, kernel, dropout)
        self.classifier = nn.Linear(80, n_classes)
        self.dropout    = nn.Dropout(dropout)
    def forward(self, x, mask):
        x = self.stage1(x, mask)
        x = self.stage2(x)
        x = self.stage3(x)
        x = x.max(dim=1).values
        x = self.dropout(x)
        return self.classifier(x)

# ── Utilities ─────────────────────────────────────────────────────────────────
class CropDataset(Dataset):
    def __init__(self, X, mask, y, indices):
        self.X    = torch.tensor(X[indices],    dtype=torch.float32)
        self.mask = torch.tensor(mask[indices], dtype=torch.float32)
        self.y    = torch.tensor(y[indices],    dtype=torch.long)
    def __len__(self):       return len(self.y)
    def __getitem__(self, i): return self.X[i], self.mask[i], self.y[i]

def compute_class_weights(y_train, n_classes):
    counts  = np.bincount(y_train, minlength=n_classes).astype(np.float32)
    weights = 1.0 / (counts + 1e-6)
    return torch.tensor(weights / weights.sum() * n_classes,
                        dtype=torch.float32).to(DEVICE)

print("✅ All functions reloaded — run Cell 7 now")

Device: cpu
✅ All functions reloaded — run Cell 7 now


In [7]:
# CELL 7 — Hyperparameter tuning to reduce overfitting
# Teacher feedback: adjust hyperparameters only, keep architecture identical
# Target: honest generalization around 90% OA

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import cohen_kappa_score, f1_score
import gc, csv

# ── Tuned hyperparameters ────────────────────────────────────────────────────
HYP_TUNED = dict(
    lr           = 0.0005,   # lower lr → slower convergence, less overfitting
    weight_decay = 1e-3,     # stronger L2 regularization (10× original)
    batch_size   = 32,
    epochs       = 200,
    n_heads      = 5,
    kernel_size  = 3,
    dropout      = 0.5,      # higher dropout (0.3→0.5)
    patience     = 15,       # stop earlier to avoid overfitting
    grad_clip    = 1.0,
    seed         = 42,
)

torch.manual_seed(HYP_TUNED['seed'])

def train_tuned(model, train_loader, val_loader, region, class_weights, hyp):
    optimizer = torch.optim.Adam(model.parameters(),
                                  lr=hyp['lr'], weight_decay=hyp['weight_decay'])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                    optimizer, T_max=hyp['epochs'])
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    best_val, patience_c = float('inf'), 0
    t_losses, v_losses   = [], []

    for epoch in range(hyp['epochs']):
        model.train()
        t_loss = 0
        for X, mask, y in train_loader:
            X, mask, y = X.to(DEVICE), mask.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(X, mask), y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), hyp['grad_clip'])
            optimizer.step()
            t_loss += loss.item()
        t_loss /= len(train_loader)

        model.eval()
        v_loss = 0
        with torch.no_grad():
            for X, mask, y in val_loader:
                X, mask, y = X.to(DEVICE), mask.to(DEVICE), y.to(DEVICE)
                v_loss += criterion(model(X, mask), y).item()
        v_loss /= len(val_loader)
        scheduler.step()
        t_losses.append(t_loss)
        v_losses.append(v_loss)

        if (epoch + 1) % 20 == 0:
            print(f"  Epoch {epoch+1:3d} | train={t_loss:.4f} | val={v_loss:.4f} | gap={v_loss-t_loss:.4f}")

        if v_loss < best_val:
            best_val, patience_c = v_loss, 0
            torch.save(model.state_dict(),
                       f'{MODEL_DIR}/new_mctnet_{region}_tuned_best.pth')
        else:
            patience_c += 1
            if patience_c >= hyp['patience']:
                print(f"  Early stopping at epoch {epoch+1}")
                break

    # Plot
    import matplotlib.pyplot as plt
    plt.figure(figsize=(8,4))
    plt.plot(t_losses, label='Train Loss')
    plt.plot(v_losses, label='Val Loss')
    plt.xlabel('Epoch'); plt.ylabel('Loss')
    plt.title(f'MCTNet TUNED — {region}')
    plt.legend(); plt.tight_layout()
    plt.savefig(f'{RES_DIR}/new_training_curve_{region}_tuned.png', dpi=150)
    plt.close()
    print(f"  Best val loss: {best_val:.4f}")


tuned_results = {}

for region, nc, class_names in [
    ('arkansas',   5, ['Soybeans','Rice','Corn','Cotton','Others']),
    ('california', 6, ['Grapes','Rice','Alfalfa','Almonds','Pistachios','Others']),
]:
    print(f"\n{'='*60}")
    print(f"TUNED MCTNet — {region.upper()}")
    print(f"  dropout={HYP_TUNED['dropout']}  weight_decay={HYP_TUNED['weight_decay']}"
          f"  lr={HYP_TUNED['lr']}  patience={HYP_TUNED['patience']}")
    print(f"{'='*60}")

    X    = np.load(f'{IN_DIR}/new_X_{region}.npy')
    mask = np.load(f'{IN_DIR}/new_mask_{region}.npy')
    y    = np.load(f'{IN_DIR}/new_y_{region}.npy')
    tr   = np.load(f'{IN_DIR}/new_train_idx_{region}.npy')
    vl   = np.load(f'{IN_DIR}/new_val_idx_{region}.npy')

    class_weights = compute_class_weights(y[tr], nc)
    print(f"  Class weights: {class_weights.cpu().numpy().round(3)}")

    train_loader = DataLoader(CropDataset(X, mask, y, tr),
                               batch_size=HYP_TUNED['batch_size'], shuffle=True)
    val_loader   = DataLoader(CropDataset(X, mask, y, vl),
                               batch_size=HYP_TUNED['batch_size'])

    model = MCTNet(n_classes=nc, seq_len=36, n_bands=10,
                   n_heads=HYP_TUNED['n_heads'],
                   kernel=HYP_TUNED['kernel_size'],
                   dropout=HYP_TUNED['dropout']).to(DEVICE)

    train_tuned(model, train_loader, val_loader, region, class_weights, HYP_TUNED)

    # Evaluate on val set
    model.load_state_dict(
        torch.load(f'{MODEL_DIR}/new_mctnet_{region}_tuned_best.pth'))
    model.eval()
    preds, labs = [], []
    with torch.no_grad():
        for X_b, mask_b, y_b in val_loader:
            out = model(X_b.to(DEVICE), mask_b.to(DEVICE))
            preds.extend(out.argmax(1).cpu().numpy())
            labs.extend(y_b.numpy())

    preds, labs = np.array(preds), np.array(labs)
    OA    = (preds == labs).mean()
    Kappa = cohen_kappa_score(labs, preds)
    F1    = f1_score(labs, preds, average='macro')

    tuned_results[region] = {'OA': OA, 'Kappa': Kappa, 'F1': F1}
    print(f"\n  Val Results (TUNED):")
    print(f"    OA={OA:.4f}  Kappa={Kappa:.4f}  F1={F1:.4f}")

    del X, mask, y; gc.collect()

# Compare original vs tuned
print(f"\n{'='*60}")
print("COMPARISON: Original vs Tuned")
print(f"{'='*60}")
original = {
    'arkansas':   {'OA': 0.9633, 'Kappa': 0.9542, 'F1': 0.9629},
    'california': {'OA': 0.9222, 'Kappa': 0.9067, 'F1': 0.9212},
}
print(f"\n{'Region':<12} {'Metric':<8} {'Original':>10} {'Tuned':>10} {'Delta':>8}")
print("-"*50)
for region in ['arkansas', 'california']:
    for metric in ['OA', 'Kappa', 'F1']:
        o = original[region][metric]
        t = tuned_results[region][metric]
        d = t - o
        print(f"{region:<12} {metric:<8} {o:>10.4f} {t:>10.4f} {'+' if d>=0 else ''}{d:>7.4f}")
    print()

# Save
with open(f'{RES_DIR}/new_mctnet_tuned_results.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['region','metric','original','tuned','delta'])
    for region in ['arkansas','california']:
        for metric in ['OA','Kappa','F1']:
            o = original[region][metric]
            t = tuned_results[region][metric]
            w.writerow([region, metric, round(o,4), round(t,4), round(t-o,4)])

print("✅ Saved → results_new/new_mctnet_tuned_results.csv")
print("→ Now re-run Phase 2 ablation with tuned model weights")


TUNED MCTNet — ARKANSAS
  dropout=0.5  weight_decay=0.001  lr=0.0005  patience=15
  Class weights: [1. 1. 1. 1. 1.]
  Epoch  20 | train=0.0381 | val=0.1447 | gap=0.1066
  Epoch  40 | train=0.0085 | val=0.1743 | gap=0.1658
  Early stopping at epoch 44
  Best val loss: 0.1292

  Val Results (TUNED):
    OA=0.9633  Kappa=0.9542  F1=0.9632

TUNED MCTNet — CALIFORNIA
  dropout=0.5  weight_decay=0.001  lr=0.0005  patience=15
  Class weights: [1. 1. 1. 1. 1. 1.]
  Epoch  20 | train=0.0620 | val=0.2302 | gap=0.1682
  Early stopping at epoch 35
  Best val loss: 0.2302

  Val Results (TUNED):
    OA=0.9417  Kappa=0.9300  F1=0.9415

COMPARISON: Original vs Tuned

Region       Metric     Original      Tuned    Delta
--------------------------------------------------
arkansas     OA           0.9633     0.9633 + 0.0000
arkansas     Kappa        0.9542     0.9542 -0.0000
arkansas     F1           0.9629     0.9632 + 0.0003

california   OA           0.9222     0.9417 + 0.0195
california   Kappa    

In [8]:
# CELL 8 — California final version retrain
# Only California, dropout=0.6, patience=10
# Saved as: new_mctnet_california_final_best.pth

HYP_FINAL_CAL = dict(
    lr           = 0.0005,
    weight_decay = 1e-3,
    batch_size   = 32,
    epochs       = 200,
    n_heads      = 5,
    kernel_size  = 3,
    dropout      = 0.6,    # more aggressive than tuned (0.5→0.6)
    patience     = 10,     # stop earlier
    grad_clip    = 1.0,
    seed         = 42,
)

torch.manual_seed(HYP_FINAL_CAL['seed'])

region      = 'california'
nc          = 6
class_names = ['Grapes','Rice','Alfalfa','Almonds','Pistachios','Others']

print(f"\n{'='*60}")
print(f"FINAL MCTNet — CALIFORNIA")
print(f"  dropout={HYP_FINAL_CAL['dropout']}  weight_decay={HYP_FINAL_CAL['weight_decay']}"
      f"  lr={HYP_FINAL_CAL['lr']}  patience={HYP_FINAL_CAL['patience']}")
print(f"{'='*60}")

X    = np.load(f'{IN_DIR}/new_X_{region}.npy')
mask = np.load(f'{IN_DIR}/new_mask_{region}.npy')
y    = np.load(f'{IN_DIR}/new_y_{region}.npy')
tr   = np.load(f'{IN_DIR}/new_train_idx_{region}.npy')
vl   = np.load(f'{IN_DIR}/new_val_idx_{region}.npy')

class_weights = compute_class_weights(y[tr], nc)
print(f"  Class weights: {class_weights.cpu().numpy().round(3)}")

train_loader = DataLoader(CropDataset(X, mask, y, tr),
                           batch_size=HYP_FINAL_CAL['batch_size'], shuffle=True)
val_loader   = DataLoader(CropDataset(X, mask, y, vl),
                           batch_size=HYP_FINAL_CAL['batch_size'])

model = MCTNet(n_classes=nc, seq_len=36, n_bands=10,
               n_heads=HYP_FINAL_CAL['n_heads'],
               kernel=HYP_FINAL_CAL['kernel_size'],
               dropout=HYP_FINAL_CAL['dropout']).to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(),
                              lr=HYP_FINAL_CAL['lr'],
                              weight_decay=HYP_FINAL_CAL['weight_decay'])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                optimizer, T_max=HYP_FINAL_CAL['epochs'])
criterion = nn.CrossEntropyLoss(weight=class_weights)

best_val, patience_c = float('inf'), 0
t_losses, v_losses   = [], []
model_path = f'{MODEL_DIR}/new_mctnet_california_final_best.pth'

for epoch in range(HYP_FINAL_CAL['epochs']):
    model.train()
    t_loss = 0
    for Xb, mb, yb in train_loader:
        Xb, mb, yb = Xb.to(DEVICE), mb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(Xb, mb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), HYP_FINAL_CAL['grad_clip'])
        optimizer.step()
        t_loss += loss.item()
    t_loss /= len(train_loader)

    model.eval()
    v_loss = 0
    with torch.no_grad():
        for Xb, mb, yb in val_loader:
            Xb, mb, yb = Xb.to(DEVICE), mb.to(DEVICE), yb.to(DEVICE)
            v_loss += criterion(model(Xb, mb), yb).item()
    v_loss /= len(val_loader)
    scheduler.step()
    t_losses.append(t_loss)
    v_losses.append(v_loss)

    if (epoch + 1) % 10 == 0:
        print(f"  Epoch {epoch+1:3d} | train={t_loss:.4f} | val={v_loss:.4f} | gap={v_loss-t_loss:.4f}")

    if v_loss < best_val:
        best_val, patience_c = v_loss, 0
        torch.save(model.state_dict(), model_path)
    else:
        patience_c += 1
        if patience_c >= HYP_FINAL_CAL['patience']:
            print(f"  Early stopping at epoch {epoch+1}")
            break

# Training curve
import matplotlib.pyplot as plt
plt.figure(figsize=(8, 4))
plt.plot(t_losses, label='Train Loss')
plt.plot(v_losses, label='Val Loss')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.title('MCTNet FINAL — california  (dropout=0.6, patience=10)')
plt.legend(); plt.tight_layout()
plt.savefig(f'{RES_DIR}/new_training_curve_california_final.png', dpi=150)
plt.close()
print(f"  Best val loss: {best_val:.4f}")

# Evaluate
model.load_state_dict(torch.load(model_path))
model.eval()
preds, labs = [], []
with torch.no_grad():
    for Xb, mb, yb in val_loader:
        out = model(Xb.to(DEVICE), mb.to(DEVICE))
        preds.extend(out.argmax(1).cpu().numpy())
        labs.extend(yb.numpy())

preds, labs = np.array(preds), np.array(labs)
OA    = (preds == labs).mean()
Kappa = cohen_kappa_score(labs, preds)
F1    = f1_score(labs, preds, average='macro')

print(f"\n  FINAL California Results:")
print(f"    OA={OA:.4f}  Kappa={Kappa:.4f}  F1={F1:.4f}")

# Compare all versions
print(f"\n{'='*55}")
print("California — All Versions Comparison")
print(f"{'='*55}")
versions = {
    'Original   (dropout=0.3, wd=1e-4, lr=1e-3)': {'OA': 0.9222, 'Kappa': 0.9067, 'F1': 0.9212},
    'Tuned      (dropout=0.5, wd=1e-3, lr=5e-4)': {'OA': 0.9417, 'Kappa': 0.9300, 'F1': 0.9415},
    'Final      (dropout=0.6, wd=1e-3, lr=5e-4)': {'OA': OA,     'Kappa': Kappa,  'F1': F1},
}
print(f"\n{'Version':<45} {'OA':>7} {'Kappa':>7} {'F1':>7}")
print("-"*68)
for name, res in versions.items():
    print(f"{name:<45} {res['OA']:>7.4f} {res['Kappa']:>7.4f} {res['F1']:>7.4f}")

# Save final summary
with open(f'{RES_DIR}/new_mctnet_final_results.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['region', 'version', 'OA', 'Kappa', 'F1',
                'dropout', 'weight_decay', 'lr', 'patience'])
    w.writerow(['arkansas',   'tuned', 0.9633, 0.9542, 0.9632,
                0.5, 0.001, 0.0005, 15])
    w.writerow(['california', 'final', round(OA,4), round(Kappa,4), round(F1,4),
                0.6, 0.001, 0.0005, 10])

print(f"\n✅ Saved → results_new/new_mctnet_final_results.csv")
print(f"   Model → models_new/new_mctnet_california_final_best.pth")
print(f"   Curve → results_new/new_training_curve_california_final.png")
print("\n→ Next: re-run Phase 2 ablation using these final model weights as baseline")

del X, mask, y; gc.collect()


FINAL MCTNet — CALIFORNIA
  dropout=0.6  weight_decay=0.001  lr=0.0005  patience=10
  Class weights: [1. 1. 1. 1. 1. 1.]
  Epoch  10 | train=0.1861 | val=0.2534 | gap=0.0673
  Epoch  20 | train=0.0928 | val=0.2625 | gap=0.1697
  Early stopping at epoch 20
  Best val loss: 0.2534

  FINAL California Results:
    OA=0.9194  Kappa=0.9033  F1=0.9209

California — All Versions Comparison

Version                                            OA   Kappa      F1
--------------------------------------------------------------------
Original   (dropout=0.3, wd=1e-4, lr=1e-3)     0.9222  0.9067  0.9212
Tuned      (dropout=0.5, wd=1e-3, lr=5e-4)     0.9417  0.9300  0.9415
Final      (dropout=0.6, wd=1e-3, lr=5e-4)     0.9194  0.9033  0.9209

✅ Saved → results_new/new_mctnet_final_results.csv
   Model → models_new/new_mctnet_california_final_best.pth
   Curve → results_new/new_training_curve_california_final.png

→ Next: re-run Phase 2 ablation using these final model weights as baseline


7484